In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Estimator-Eingaben und -Ausgaben

<Accordion>
<AccordionItem title="Paketversionen">

Der Code auf dieser Seite wurde mit den folgenden Anforderungen entwickelt.
Wir empfehlen, diese Versionen oder neuere zu verwenden.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Diese Seite gibt einen Überblick über die Eingaben und Ausgaben des Qiskit Runtime Estimator-Primitives, das Workloads auf IBM Quantum&reg; Rechenressourcen ausführt. Der Estimator ermöglicht es dir, vektorisierte Workloads effizient zu definieren, indem du eine Datenstruktur namens [**Primitive Unified Bloc (PUB)**](/guides/primitive-input-output#pubs) verwendest. Diese werden als Eingaben für die [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run)-Methode des Estimator-Primitives verwendet, die den definierten Workload als Job ausführt. Nachdem der Job abgeschlossen ist, werden die Ergebnisse in einem Format zurückgegeben, das sowohl von den verwendeten PUBs als auch von den vom Primitive angegebenen Laufzeitoptionen abhängt.
## Eingaben
Jeder PUB hat dieses Format:

(`<einzelner Circuit>`, `<ein oder mehrere Observablen>`, `<optionale ein oder mehrere Parameterwerte>`, `<optionale Präzision>`),

Die optionalen `Parameterwerte` können eine Liste oder ein einzelner Parameter sein. Elemente aus Observablen und Parameterwerten werden nach den NumPy-Broadcasting-Regeln kombiniert, wie im Thema [Primitive-Eingaben und -Ausgaben](primitive-input-output#broadcasting-rules) beschrieben, und für jedes Element der gesendeten Form wird ein Schätzwert für den Erwartungswert zurückgegeben.

> **Note:** Wenn die Eingabe Messungen enthält, werden diese ignoriert.

Für das Estimator-Primitive kann ein PUB höchstens vier Werte enthalten:
- Einen einzelnen `QuantumCircuit`, der ein oder mehrere [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)-Objekte enthalten kann
- Eine Liste von einer oder mehreren Observablen, die die zu schätzenden Erwartungswerte angeben, in ein Array angeordnet (zum Beispiel eine einzelne Observable als 0-dimensionales Array, eine Liste von Observablen als 1-dimensionales Array usw.). Die Daten können in einem der `ObservablesArrayLike`-Formate wie `Pauli`, `SparsePauliOp`, `PauliList` oder `str` vorliegen.
   > **Note:** - Kommutierende Observablen **im selben PUB** werden mit [dieser Methode](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting) gruppiert.
>    - Kommutierende Observablen in verschiedenen PUBs, auch wenn sie denselben Circuit haben, werden nicht mit derselben Messung geschätzt. Jeder PUB stellt eine andere Messbasis dar, daher sind für jeden PUB separate Messungen erforderlich.
>    - Um sicherzustellen, dass kommutierende Observablen mit derselben Messung geschätzt werden, gruppiere sie innerhalb desselben PUBs.
- Eine Sammlung von Parameterwerten, gegen die der Circuit gebunden wird. Dies kann als einzelnes array-ähnliches Objekt angegeben werden, bei dem der letzte Index über Circuit-`Parameter`-Objekte geht, oder weggelassen (oder äquivalent auf `None` gesetzt) werden, wenn der Circuit keine `Parameter`-Objekte hat.
- (Optional) Eine Zielpräzision für die zu schätzenden Erwartungswerte
---

Der folgende Code zeigt ein Beispiel für einen Satz vektorisierter Eingaben für das `Estimator`-Primitive und führt diese auf einem IBM&reg; Backend als einzelnes `RuntimeJobV2`-Objekt aus.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
)

import numpy as np

# Instantiate runtime service and get
# the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 100),
        np.linspace(-4 * np.pi, 4 * np.pi, 100),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator_pub = (transpiled_circuit, observables, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
estimator = Estimator(mode=backend)
job = estimator.run([estimator_pub])
result = job.result()

## Ausgaben
Nachdem ein oder mehrere PUBs zur Ausführung an eine QPU gesendet wurden und ein Job erfolgreich abgeschlossen wurde, werden die Daten als [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult)-Container-Objekt zurückgegeben, auf das durch Aufrufen der `RuntimeJobV2.result()`-Methode zugegriffen wird.

Das `PrimitiveResult` enthält eine iterierbare Liste von [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult)-Objekten, die die Ausführungsergebnisse für jeden PUB enthalten.

Jedes Element dieser Liste entspricht jedem PUB, der an die `run()`-Methode des Primitives übergeben wurde (zum Beispiel gibt ein Job, der mit 20 PUBs übermittelt wurde, ein `PrimitiveResult`-Objekt zurück, das eine Liste von 20 `PubResult`-Objekten enthält, eines für jeden PUB).

Jedes `PubResult` für das Estimator-Primitive enthält mindestens ein Array von Erwartungswerten (`PubResult.data.evs`) und zugehörige Standardabweichungen (entweder `PubResult.data.stds` oder `PubResult.data.ensemble_standard_error`, je nach verwendetem `resilience_level`), kann aber je nach den angegebenen Fehlerminderungsoptionen mehr Daten enthalten.

Jedes `PubResult`-Objekt besitzt sowohl ein `data`- als auch ein `metadata`-Attribut.
    - Das `data`-Attribut ist ein angepasster [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin), der die tatsächlichen Messwerte, Standardabweichungen usw. enthält.
    - Der `DataBin` hat verschiedene Attribute je nach Form oder Struktur des zugehörigen PUBs sowie den Fehlerminderungsoptionen, die durch das Primitive angegeben wurden, das zum Übermitteln des Jobs verwendet wurde (zum Beispiel [ZNE](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) oder [PEC](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)).
    - Das `metadata`-Attribut enthält Informationen über die verwendeten Laufzeit- und Fehlerminderungsoptionen (später im Abschnitt [Ergebnis-Metadaten](#result-metadata) dieser Seite erklärt).

Das Folgende ist ein visueller Überblick über die `PrimitiveResult`-Datenstruktur für die Estimator-Ausgabe:

    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```

Kurz gesagt gibt ein einzelner Job ein `PrimitiveResult`-Objekt zurück und enthält eine Liste von einem oder mehreren `PubResult`-Objekten. Diese `PubResult`-Objekte speichern dann die Messdaten für jeden PUB, der dem Job übergeben wurde.

Der folgende Code-Ausschnitt beschreibt das `PrimitiveResult`- (und zugehörige `PubResult`-)Format für den oben erstellten Job.

In [ ]:
print(
    f"The result of the submitted job had {len(result)} "
    f"PUBs and has a value:\n {result}\n"
)
print(
    "The associated PubResult of this job has the following data bins:\n "
    "{result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets"
    "having shape (100, 2), where 2 is the number of parameters in the "
    "circuit, combined with our array of observables having shape (3, 1). \n"
)
with np.printoptions(threshold=200):
    print(
        "The expectation values measured from this PUB are: \n"
        "{result[0].data.evs}\n"
    )

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=(3, 100), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(3, 100), dtype=float64>), shape=(3, 100)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=

#### How the Estimator primitive calculates error

In addition to the estimate of the mean of the observables passed in the input PUBs (the `evs` field of the `DataBin`), Estimator also attempts to deliver an estimate of the error associated with those expectation values. All Estimator queries will populate the `stds` field with a quantity like the standard error of the mean for each expectation value, but some error mitigation options produce additional information, such as `ensemble_standard_error`.

Consider a single observable $\mathcal{O}$. In the absence of [ZNE](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne), you can think of each shot of the Estimator execution as providing a point estimate of the expectation value $\langle \mathcal{O} \rangle$. If the pointwise estimates are in a vector `Os`, then the value returned in `ensemble_standard_error` is equivalent to the following (in which $\sigma_{\mathcal{O}}$ is the [standard deviation of the expectation value](/docs/api/qiskit/qiskit.primitives.BackendEstimatorV2) estimate and $N_{shots}$ is the number of shots):

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{shots}} },$$

which treats all shots as part of a single ensemble. If you requested gate [twirling](/docs/guides/error-mitigation-and-suppression-techniques#pauli-twirling) (`twirling.enable_gates = True`), you can sort the pointwise estimates of $\langle \mathcal{O} \rangle$ into sets that share a common twirl. Call these sets of estimates `O_twirls`, and there are `num_randomizations` (number of twirls) of them. Then `stds` is the standard error of the mean of `O_twirls`, as in

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{twirls}} },$$

where $\sigma_{\mathcal{O}}$ is the standard deviation of `O_twirls` and $N_{twirls}$ is the number of twirls. When you do not enable twirling, `stds` and `ensemble_standard_error` are equal.

If you enable ZNE, then the `stds` described above become weights in a non-linear regression to an extrapolator model. What finally gets returned in the `stds` in this case is the uncertainty of the fit model evaluated at a noise factor of zero. When there is a poor fit, or large uncertainty in the fit, the reported `stds` can become very large. When ZNE is enabled, `pub_result.data.evs_noise_factors` and `pub_result.data.stds_noise_factors` are also populated, so that you can do your own extrapolation.

## Result metadata

In addition to the execution results, both the `PrimitiveResult` and `PubResult` objects contain a metadata attribute about the job that was submitted. The metadata containing information for all submitted PUBs (such as the various [runtime options](/docs/api/qiskit-ibm-runtime/options) available) can be found in the `PrimitiveResult.metatada`, while the metadata specific to each PUB is found in `PubResult.metadata`.

<Admonition type="note">
In the metadata field, primitive implementations can return any information about execution that is relevant to them, and there are no key-value pairs that are guaranteed by the base primitive. Thus, the returned metadata might be different in different primitive implementations.
</Admonition>

In [3]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'dynamical_decoupling' : {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'},
'twirling' : {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'},
'resilience' : {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False},
'version' : 2,

The metadata of the PubResult result is:
'shots' : 4096,
'target_precision' : 0.015625,
'circuit_metadata' : {},
'resilience' : {},
'num_randomizations' : 32,
